In [ ]:
import os
import json
from dataclasses import dataclass, field
from typing import List, Optional

from openai import OpenAI
from qdrant_client import QdrantClient

client = OpenAI(api_key="")  # берет ключ из ENV

EMBED_MODEL = "text-embedding-3-large"
CHAT_MODEL = "gpt-4o-mini"

QDRANT_URL = "https://qdrant.dev.adapstory.com"
COLLECTION_NAME = "presentations_industrix_openai"

os.environ["QDRANT_DISABLE_CHECK"] = "1"

qdrant = QdrantClient(url=QDRANT_URL, port=443, timeout=120)


# ─────────────────────────── helpers ────────────────────────────

def embed_query(text: str) -> list:
    return client.embeddings.create(model=EMBED_MODEL, input=text).data[0].embedding


def ask_llm(prompt: str, temperature: float = 0.3) -> str:
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=700,
    )
    return response.choices[0].message.content.strip()


def parse_json(text: str) -> dict:
    """Безопасный парсинг JSON — убирает markdown-обёртки."""
    cleaned = text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(cleaned)


def retrieve_context(question: str, top_k: int = 10) -> list:
    vector = embed_query(question)
    results = qdrant.search(
        collection_name=COLLECTION_NAME,
        query_vector=vector,
        limit=top_k,
        with_payload=True,
    )
    contexts = []
    for r in results:
        payload = r.payload or {}
        text = payload.get("text", "").strip()
        if not text:
            continue
        contexts.append({
            "text": text,
            "page": payload.get("page"),
            "source": payload.get("source"),
            "score": r.score,
        })
    return contexts


def build_context(contexts: list) -> str:
    chunks = []
    for i, c in enumerate(contexts):
        chunks.append(
            f"CHUNK_ID: {i}\nSOURCE: {c.get('source')}\nPAGE: {c.get('page')}\n\nTEXT:\n{c.get('text')}"
        )
    return "\n\n---\n\n".join(chunks)


def format_history(history: list) -> str:
    return "\n".join(history) if history else "История диалога пуста."


# ─────────────────────────── state ──────────────────────────────

@dataclass
class StudentState:
    topic: str
    attempts: int = 0
    mastery_score: float = 0.0
    history: List[str] = field(default_factory=list)
    misconceptions: List[str] = field(default_factory=list)
    last_strategy: Optional[str] = None
    explanation_used: bool = False
    consecutive_wrong: int = 0
    consecutive_chat: int = 0   # сколько раз подряд студент уклонился от ответа


# ─────────────────────────── question type ──────────────────────
# НОВОЕ: классификатор типа вопроса — learning vs organizational

class QuestionTypeClassifier:
    """
    Определяет тип вопроса ДО запуска учебной сессии.

    organizational — вопрос про процесс курса (домашки, дедлайны,
                     обратная связь, расписание, оценки и т.п.)
    learning       — вопрос про содержание курса (термины, концепции)
    other          — всё остальное
    """

    def classify(self, question: str) -> str:
        prompt = f"""
Ты классифицируешь вопросы студентов в учебном курсе.

Вопрос: "{question}"

Категории и их ПРИЗНАКИ:

organizational — вопрос про ПРОЦЕСС обучения, а НЕ про содержание.
  Ключевые слова: домашка, задание, дедлайн, обратная связь, преподаватель,
  расписание, оценки, платформа, проверка, сдача, срок, вебинар, чат, ментор, эксперт, формат сдачи, домашенее задание, как сдвать.
  Примеры: "Будет ли обратная связь?", "Когда дедлайн?",
            "Как сдавать домашку?", "Есть ли проверка заданий?"

learning — вопрос про содержание курса: термины, концепции, темы.
  Примеры: "Что такое юнит-экономика?", "Почему стартапы проваливаются?"

other — всё остальное, не связанное с курсом.

ЖЁСТКОЕ ПРАВИЛО: если в вопросе есть хотя бы одно из слов:
домашка / задание / дедлайн / обратная связь / преподаватель /
расписание / проверка / вебинар / чат — это ВСЕГДА organizational.

Верни ТОЛЬКО JSON без пояснений: {{"type": "learning|organizational|other"}}
"""
        try:
            res = ask_llm(prompt, temperature=0)
            return parse_json(res).get("type", "learning")
        except Exception:
            return "learning"


class OrganizationalAgent:
    """
    Отвечает на организационные вопросы ТОЛЬКО из контекста RAG.
    Не придумывает ответы из общих знаний.
    """

    def respond(self, question: str, context: str) -> str:
        prompt = f"""
Студент курса задал организационный вопрос.

Информация из материалов курса:
{context}

Вопрос студента: "{question}"

Правила:
- Отвечай ТОЛЬКО на основе информации из материалов курса выше
- Если информация есть — изложи её кратко и точно своими словами (2-4 предложения)
- Если информации нет — скажи: "В материалах курса нет информации по этому вопросу. Уточните у преподавателя."
- Не придумывай ответы из общих знаний LLM
"""
        return ask_llm(prompt, temperature=0.1)


# ─────────────────────────── agents ─────────────────────────────

class PlannerAgent:
    """
    Стратегии:
      socratic  — задаём наводящий вопрос
      hint      — даём подсказку
      explain   — объясняем тему
      verify    — после объяснения проверяем понимание
    """

    def plan(
        self,
        question: str,
        context_text: str,
        state: StudentState,
        last_answer: Optional[str] = None,
        evaluation: Optional[dict] = None,
    ) -> dict:

        misconceptions_text = (
            "\n".join(f"- {m}" for m in state.misconceptions)
            if state.misconceptions
            else "нет"
        )

        prompt = f"""
Ты педагогический планировщик ИИ-тьютора.

Выбери ОДНУ стратегию из: socratic, hint, explain, verify.

Правила выбора:
1. socratic  — студент ещё не пробовал отвечать или ответил частично (score < 0.5), attempts <= 1
2. hint      — студент ответил, но score < 0.6, attempts <= 3; или есть явное заблуждение
3. explain   — attempts >= 3 ИЛИ consecutive_wrong >= 2 ИЛИ студент явно не понимает
4. verify    — стратегия была explain → нужно проверить понимание (задать короткий вопрос)
5. НЕ повторяй одну стратегию больше 2 раз подряд без изменений
6. Если score > 0.8 → ответь "mastered" (тема усвоена, сессию можно завершать)

Текущее состояние:
- attempts: {state.attempts}
- consecutive_wrong: {state.consecutive_wrong}
- last_strategy: {state.last_strategy or "нет"}
- explanation_used: {state.explanation_used}
- mastery_score: {state.mastery_score}

Заблуждения студента:
{misconceptions_text}

Последний ответ студента:
{last_answer or "пока нет ответа"}

Оценка последнего ответа:
{json.dumps(evaluation, ensure_ascii=False) if evaluation else "нет"}

Контекст (фрагмент):
{context_text[:800]}

Верни ТОЛЬКО JSON без пояснений:
{{"strategy": "socratic|hint|explain|verify|mastered"}}
"""

        response = ask_llm(prompt, temperature=0.1)

        try:
            cleaned = response.strip().removeprefix("```json").removesuffix("```").strip()
            plan = json.loads(cleaned)
            strategy = plan.get("strategy", "socratic")
        except Exception:
            strategy = "socratic"

        return {"strategy": strategy}


class TutorAgent:

    def respond(
        self,
        strategy: str,
        question: str,
        contexts: list,
        history: list,
        misconceptions: Optional[List[str]] = None,
    ) -> str:

        context_text = build_context(contexts)
        history_text = format_history(history)
        misc_text = (
            "Заблуждения студента, которые нужно исправить:\n"
            + "\n".join(f"- {m}" for m in misconceptions)
            if misconceptions
            else ""
        )

        strategy_instructions = {
            "socratic": (
                "Задай ОДИН наводящий вопрос, который помогает студенту самому прийти к ответу.\n"
                "Не давай ответ и не перечисляй факты. Максимум — 2 предложения."
            ),
            "hint": (
                "Дай ОДНУ небольшую подсказку, указывающую направление мысли.\n"
                "Не раскрывай полный ответ. Упомяни заблуждение студента, если оно есть.\n"
                "Максимум — 3 предложения."
            ),
            "explain": (
                "Объясни тему развёрнуто и структурированно.\n"
                "Используй ТОЛЬКО контекст курса.\n"
                "В конце укажи источник: [Источник: SOURCE, стр. PAGE]\n"
                "Исправь заблуждения студента, если они есть."
            ),
            "verify": (
                "Ты только что объяснил тему. Теперь задай студенту ОДИН короткий проверочный вопрос,\n"
                "чтобы убедиться, что он понял объяснение. Вопрос должен быть конкретным."
            ),
        }

        instruction = strategy_instructions.get(strategy, strategy_instructions["socratic"])

        prompt = f"""
Ты ИИ-тьютор курса. Используй ТОЛЬКО информацию из контекста.
Если информации нет — скажи: «В материалах курса нет информации по этому вопросу».

КОНТЕКСТ КУРСА:
{context_text}

ВОПРОС СТУДЕНТА:
{question}

ИСТОРИЯ ДИАЛОГА:
{history_text}

{misc_text}

СТРАТЕГИЯ: {strategy}
ИНСТРУКЦИЯ:
{instruction}

ОТВЕТ ТЬЮТОРА:
"""

        return ask_llm(prompt, temperature=0.3)


class IntentClassifier:
    """
    Определяет намерение сообщения студента:
      answer    — попытка ответить на вопрос тьютора
      chat      — разговорное сообщение (спасибо, понял, окей и т.д.)
      question  — студент задаёт свой вопрос по теме
      off_topic — сообщение не по теме
    """

    def classify(self, student_message: str, tutor_last_message: str, topic: str) -> str:

        prompt = f"""
Ты классификатор намерений в диалоге тьютора и студента.

Тема обучения: {topic}

Последнее сообщение тьютора:
{tutor_last_message}

Сообщение студента:
{student_message}

Определи намерение сообщения студента. Варианты:
- answer     — студент пытается ответить на вопрос тьютора (даже частично или неверно)
- chat       — разговорное сообщение: благодарность, подтверждение, "понял", "окей", "спасибо" и т.п.
- question   — студент задаёт свой вопрос по теме курса
- off_topic  — сообщение не связано с темой обучения

Верни ТОЛЬКО JSON:
{{"intent": "answer|chat|question|off_topic"}}
"""

        response = ask_llm(prompt, temperature=0)

        try:
            cleaned = response.strip().removeprefix("```json").removesuffix("```").strip()
            return json.loads(cleaned).get("intent", "answer")
        except Exception:
            return "answer"


class ChatAgent:
    """
    Отвечает на разговорные сообщения и вопросы студента,
    после чего мягко возвращает к теме обучения.
    """

    def respond_to_chat(
        self,
        student_message: str,
        tutor_last_message: str,
        topic: str,
        history: list,
    ) -> str:

        history_text = format_history(history[-6:])

        prompt = f"""
Ты ИИ-тьютор курса. Студент написал тебе разговорное сообщение.

Тема обучения: {topic}

История диалога:
{history_text}

Твоё последнее сообщение:
{tutor_last_message}

Сообщение студента:
{student_message}

Ответь естественно и коротко (1-2 предложения).
Затем мягко напомни свой предыдущий вопрос или предложи продолжить.
Не повторяй весь вопрос дословно — перефразируй кратко.
"""

        return ask_llm(prompt, temperature=0.4)

    def respond_to_question(
        self,
        student_question: str,
        contexts: list,
        topic: str,
        history: list,
        tutor_last_message: str,
    ) -> str:

        context_text = build_context(contexts[:3])
        history_text = format_history(history[-6:])

        prompt = f"""
Ты ИИ-тьютор курса. Студент задал уточняющий вопрос.

Тема обучения: {topic}
Контекст курса: {context_text}

История диалога:
{history_text}

Вопрос студента: {student_question}

Ответь кратко, используя только контекст курса (2-4 предложения).
Если информации нет — скажи честно.
После ответа мягко верни студента к теме, напомнив свой вопрос одним предложением.

Твой предыдущий вопрос был: {tutor_last_message}
"""

        return ask_llm(prompt, temperature=0.3)


class EvaluatorAgent:

    def evaluate(self, question: str, student_answer: str, context: str) -> dict:

        prompt = f"""
Ты оцениваешь ответ студента на вопрос по материалам курса.

Вопрос: {question}
Ответ студента: {student_answer}
Контекст курса: {context}

Верни ТОЛЬКО JSON без пояснений:
{{
  "correct": true/false,
  "partial": true/false,
  "misconception": "описание заблуждения или пустая строка",
  "score": 0.0-1.0,
  "feedback": "краткий комментарий для тьютора (не для студента)"
}}
"""

        response = ask_llm(prompt, temperature=0)

        try:
            cleaned = response.strip().removeprefix("```json").removesuffix("```").strip()
            return json.loads(cleaned)
        except Exception:
            return {"correct": False, "partial": False, "misconception": "", "score": 0.0, "feedback": ""}


# ─────────────────────────── orchestrator ───────────────────────

class TutorOrchestrator:

    MAX_ATTEMPTS = 6

    def __init__(self, question: str):
        self.question = question
        self.contexts = retrieve_context(question)
        self.context_text = build_context(self.contexts[:5])

        self.state = StudentState(topic=question)

        # НОВОЕ: агенты для организационных вопросов
        self.qtype_clf = QuestionTypeClassifier()
        self.org_agent = OrganizationalAgent()

        self.planner = PlannerAgent()
        self.tutor = TutorAgent()
        self.evaluator = EvaluatorAgent()
        self.intent_clf = IntentClassifier()
        self.chat_agent = ChatAgent()

        self._last_tutor_msg = ""

    def _update_state_after_eval(self, evaluation: dict, last_answer: str):
        score = evaluation.get("score", 0.0)
        self.state.mastery_score = score
        self.state.attempts += 1

        misconception = evaluation.get("misconception", "")
        if misconception and misconception not in self.state.misconceptions:
            self.state.misconceptions.append(misconception)

        if not evaluation.get("correct") and score < 0.5:
            self.state.consecutive_wrong += 1
        else:
            self.state.consecutive_wrong = 0

        self.state.history.append(f"Студент: {last_answer}")

    def _print_sep(self):
        print("\n" + "─" * 50)

    def run(self):

        # ══════════════════════════════════════════
        # ШАГ 0: ОПРЕДЕЛЯЕМ ТИП ВОПРОСА
        # ══════════════════════════════════════════

        qtype = self.qtype_clf.classify(self.question)
        print(f"🔎 Тип вопроса: {qtype}")

        # Организационный вопрос → отвечаем из RAG и завершаем
        if qtype == "organizational":
            if not self.contexts:
                print("\n🧠 Тьютор: В материалах курса нет информации по этому вопросу. Уточните у преподавателя.")
            else:
                print("\n🧠 Тьютор:", self.org_agent.respond(self.question, self.context_text))
            print("\n✅ Завершено")
            return

        # Учебный вопрос — проверяем базу знаний
        if not self.contexts:
            print("❌ Нет информации в базе знаний по данному вопросу.")
            return

        print("\n📚 Начинаем обучение\n")
        print(f"📌 Тема: {self.question}\n")

        last_answer = None
        evaluation = None
        force_next_strategy = None

        while True:

            # ════════════════════════════════════════════
            # ФАЗА 1: ТЬЮТОР ГОВОРИТ
            # ════════════════════════════════════════════

            if force_next_strategy:
                strategy = force_next_strategy
                force_next_strategy = None
            else:
                plan = self.planner.plan(
                    self.question,
                    self.context_text,
                    self.state,
                    last_answer,
                    evaluation,
                )
                strategy = plan["strategy"]

            if strategy == "mastered":
                print("\n✅ Отлично! Тьютор подтверждает: тема усвоена.")
                break

            if self.state.attempts >= self.MAX_ATTEMPTS:
                print("\n⚠️  Достигнут лимит попыток. Завершаем сессию.")
                break

            self._print_sep()
            print(f"📊 Стратегия: {strategy}")

            tutor_msg = self.tutor.respond(
                strategy,
                self.question,
                self.contexts,
                self.state.history,
                self.state.misconceptions,
            )
            print(f"\n🧠 Тьютор: {tutor_msg}")

            self.state.last_strategy = strategy
            self.state.history.append(f"Тьютор: {tutor_msg}")
            self._last_tutor_msg = tutor_msg

            # После explain → следующий ход ВСЕГДА verify (детерминировано)
            if strategy == "explain":
                self.state.explanation_used = True
                force_next_strategy = "verify"

            # ════════════════════════════════════════════
            # ФАЗА 2: СТУДЕНТ ГОВОРИТ
            # (внутренний цикл — chat/question/off_topic крутятся здесь)
            # ════════════════════════════════════════════

            while True:
                print()
                raw_input = input("👨‍🎓 Студент: ").strip()

                if not raw_input:
                    print("(тьютор ждёт ответа...)")
                    continue

                intent = self.intent_clf.classify(
                    raw_input,
                    self._last_tutor_msg,
                    self.question,
                )
                print(f"💬 Намерение: {intent}")

                # chat ─────────────────────────────────────────
                if intent == "chat":
                    reply = self.chat_agent.respond_to_chat(
                        raw_input,
                        self._last_tutor_msg,
                        self.question,
                        self.state.history,
                    )
                    print(f"\n🧠 Тьютор: {reply}")

                    # После explain/verify "спасибо" = сессия завершена
                    if self.state.explanation_used:
                        print("\n✅ Сессия завершена. Удачи в обучении!")
                        return

                    self.state.history.append(f"Студент: {raw_input}")
                    self.state.history.append(f"Тьютор: {reply}")
                    self._last_tutor_msg = reply
                    self.state.consecutive_chat += 1

                    # Студент 2 раза уклонился — принудительно двигаемся дальше
                    if self.state.consecutive_chat >= 2:
                        self.state.consecutive_chat = 0
                        self.state.consecutive_wrong += 1  # засчитываем как неудачную попытку
                        print("\n🧠 Тьютор: Похоже, тема даётся сложно — давай я объясню сам.")
                        # Выходим из фазы 2, планировщик выберет explain
                        break

                    continue

                # question ─────────────────────────────────────
                if intent == "question":
                    reply = self.chat_agent.respond_to_question(
                        raw_input,
                        self.contexts,
                        self.question,
                        self.state.history,
                        self._last_tutor_msg,
                    )
                    print(f"\n🧠 Тьютор: {reply}")
                    self.state.history.append(f"Студент: {raw_input}")
                    self.state.history.append(f"Тьютор: {reply}")
                    self._last_tutor_msg = reply
                    continue

                # off_topic ────────────────────────────────────
                if intent == "off_topic":
                    print(f"\n🧠 Тьютор: Давай вернёмся к теме. {self._last_tutor_msg}")
                    continue

                # answer → выходим из фазы 2 ──────────────────
                break

            # ════════════════════════════════════════════
            # ФАЗА 3: ОЦЕНИВАЕМ ОТВЕТ
            # ════════════════════════════════════════════

            last_answer = raw_input

            evaluation = self.evaluator.evaluate(
                self.question,
                last_answer,
                self.context_text,
            )

            score = evaluation.get("score", 0.0)
            feedback = evaluation.get("feedback", "")
            print(f"\n📊 Оценка: score={score:.2f} | {feedback}")

            self._update_state_after_eval(evaluation, last_answer)

            if evaluation.get("correct") or score >= 0.85:
                print("\n✅ Правильно! Тема усвоена.")
                break


# ─────────────────────────── entry point ────────────────────────

if __name__ == "__main__":
    question = "Какие вообще модули и этапы курса"
    TutorOrchestrator(question).run()